In [ ]:
import pandas as pd
from pathlib import Path
from reed import utils
import tarfile

In [ ]:
config = utils.Config()

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"

## WWR

In [ ]:
wwr_pmg11007 = monitoring_data_path/'WWR - PMG-11007'

In [ ]:
files_with_energy_data = {}

for tar_path in wwr_pmg11007.glob("*.tar.gz"):
    try:
        with tarfile.open(tar_path) as tar:
            member = tar.getmembers()[0]  # assume one CSV inside
            # read only first few rows to inspect columns
            df_preview = pd.read_csv(tar.extractfile(member), nrows=5)
            files_with_energy_data[tar_path.name] = "Power_Active" in df_preview.columns
    except Exception as e:
        files_with_energy_data[tar_path.name] = f"Error: {e}"

files_with_energy_data

In [ ]:
def load_wwr(file_number):
    paths = list(wwr_pmg11007.glob("*.gz"))
    wwr_path = paths[file_number]

    wwr_df = pd.read_csv(wwr_path)
    wwr_df["timestamp"] = pd.to_datetime(wwr_df["timestamp"])

    return wwr_df

In [ ]:
keep_columns = ["Tool_Number", "Power_Active"]

In [ ]:
# operation OF10001 - WWR
file001 = load_wwr(0)
file002 = load_wwr(1) # has energy data
file003 = load_wwr(2)
file004 = load_wwr(3) # has tool data

In [ ]:
# operation OF10002 - WWR
file005 = load_wwr(4)
file006 = load_wwr(5) # has energy data
file007 = load_wwr(6)
file008 = load_wwr(7) # has tool data

In [ ]:
file006.info()

In [ ]:
# sort the relevant files by timestamp
file006 = file006.sort_values("timestamp")
file008 = file008.sort_values("timestamp")

In [ ]:
# merge df for file002 and file004
WWR_OF10002_df = pd.merge_asof(file006, file008, on="timestamp", direction="nearest", tolerance=pd.Timedelta("300ms"))
WWR_OF10002_df.head(20)

In [ ]:
WWR_OF10002_df.sort_values("timestamp")

In [ ]:
def analyse_wwr(wwr_df, keep_columns):
    df = wwr_df
    df = df[keep_columns].copy()
    df["energy_consumption_kWh"] = df["Power_Active"] / 3600
    df_bytool = (
        df.groupby("Tool_Number", as_index=False)
            .agg({
                "energy_consumption_kWh": "sum",
                "Power_Active": "mean"
            })
        )

    return df_bytool

In [ ]:
WWR_OF10002_df_bytool = analyse_wwr(WWR_OF10002_df, keep_columns)
WWR_OF10002_df_bytool

In [ ]:
WWR_OF10002_df_bytool["energy_consumption_kWh"].sum()

In [ ]:
# tools change over time
import matplotlib.pyplot as plt

def plot_energy_vs_time(*dfs, label_prefix="WWR_OF10002_df"):
    plt.figure(figsize=(14,6))

    for i, df in enumerate(dfs, start=1):
        # calculate elapsed time since start of dataset
        df = df.copy()
        df["energy_consumption"] = df["Power_Active"] / 3600
        df["elapsed_seconds"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        
        # plot meaningfully, fx energy_consumption (or Tool_Number)
        plt.plot(df["elapsed_seconds"], df["energy_consumption"], label=f"{label_prefix}{i}")

    plt.title("Energy consumption over elapsed time")
    plt.xlabel("Time since start (seconds)")
    plt.ylabel("Energy consumption (kWh per second)")
    plt.legend()
    plt.show()


# --- USE IT ---
plot_energy_vs_time(WWR_OF10002_df)

